<a href="https://colab.research.google.com/github/kyahikaru/hinglish-prompt-injection-detector/blob/main/notebooks/srinivasan_dataset_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install sentence-transformers scikit-learn pandas numpy openpyxl -q

In [2]:
from google.colab import files
print("Upload these two files:")
print("1. PromptInjectionPrompts.xlsx (Srinivasan dataset)")
print("2. adversarial_test_set_v2.json (250 stealth samples)")
uploaded = files.upload()

Upload these two files:
1. PromptInjectionPrompts.xlsx (Srinivasan dataset)
2. adversarial_test_set_v2.json (250 stealth samples)


Saving adversarial_test_set_v2.json to adversarial_test_set_v2.json
Saving srinivasan_dataset.xlsx to srinivasan_dataset.xlsx


In [13]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_excel('srinivasan_dataset.xlsx')
df = df.rename(columns={'Prompts': 'text', 'Label': 'label'})

train_df, test_df = train_test_split(df, test_size=0.20, stratify=df['label'], random_state=42)

print("--- SRINIVASAN DATASET SPLIT ---")
print(f"Train samples: {len(train_df)}")
print(f"Test samples : {len(test_df)}")

--- SRINIVASAN DATASET SPLIT ---
Train samples: 3200
Test samples : 800


In [15]:
from sentence_transformers import SentenceTransformer

# === CORRECT MODEL FOR SRINIVASAN BASELINE (L12) ===
model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')

X_train = model.encode(train_df['text'].tolist(), show_progress_bar=True)
X_test  = model.encode(test_df['text'].tolist(), show_progress_bar=True)

y_train = train_df['label'].values
y_test  = test_df['label'].values

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

clf = LogisticRegression(max_iter=1000, class_weight='balanced')
clf.fit(X_train, y_train)

print("--- SRINIVASAN BASELINE METRICS ON SRINIVASAN TEST SET ---")
print(classification_report(y_test, clf.predict(X_test)))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/100 [00:00<?, ?it/s]

Batches:   0%|          | 0/25 [00:00<?, ?it/s]

--- SRINIVASAN BASELINE METRICS ON SRINIVASAN TEST SET ---
              precision    recall  f1-score   support

           0       0.94      0.94      0.94       400
           1       0.94      0.94      0.94       400

    accuracy                           0.94       800
   macro avg       0.94      0.94      0.94       800
weighted avg       0.94      0.94      0.94       800



In [16]:
# === UPDATED: Test Srinivasan baseline on 250-sample adversarial set ===

import json

print("Loading 250-sample adversarial test set (v3)...")
with open('adversarial_test_set_v2.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

samples = data['samples'] if 'samples' in data else data
print(f"Loaded {len(samples)} stealth adversarial prompts successfully\n")

print("=== SRINIVASAN BASELINE vs 250 STEALTH SAMPLES ===")

blocked = 0
results = []

for sample in samples:
    text = sample['hinglish']
    emb = model.encode([text], convert_to_numpy=True).astype('float32')
    pred = clf.predict(emb)[0]
    status = "BLOCKED" if pred == 1 else "BYPASSED"
    if pred == 1:
        blocked += 1
    results.append({
        "id": sample["id"],
        "category": sample["category"],
        "prompt": text[:80] + ("..." if len(text) > 80 else ""),
        "srinivasan": status
    })

summary_df = pd.DataFrame(results)
print(summary_df.groupby("category").size().reset_index(name="count"))

print(f"\nFINAL RESULT:")
print(f"Srinivasan-style model blocked : {blocked}/250 ({blocked/250*100:.1f}%)")
print(f"Bypassed (failed)              : {250 - blocked}/250")

Loading 250-sample adversarial test set (v3)...
Loaded 250 stealth adversarial prompts successfully

=== SRINIVASAN BASELINE vs 250 STEALTH SAMPLES ===
                   category  count
0          academic_masking     24
1   authority_impersonation     23
2          direct_jailbreak      6
3           fiction_framing     31
4        gradual_escalation     24
5              mixed_script     23
6                obfuscated     20
7              pure_english     33
8                pure_hindi     33
9          research_framing     29
10         stealth_academic      1
11          stealth_fiction      1
12         stealth_research      2

FINAL RESULT:
Srinivasan-style model blocked : 211/250 (84.4%)
Bypassed (failed)              : 39/250
